# ILQX based agents

> Agents utelizing the ILQX approach from https://doi.org/10.1287/mnsc.2020.3680 for Dynamic pricing and learning problems

#| default_exp agents.dynamic_pricing.ILQX

In [ ]:

#from nbdev.showdoc import *

In [ ]:


import logging

from abc import ABC, abstractmethod
from typing import Union, Optional, List
import numpy as np
import joblib   
import os
from ddopai.agents.dynamic_pricing.utils import GLMLink
from ddopai.envs.base import BaseEnvironment
from ddopai.agents.base import BaseAgent
from ddopai.utils import MDPInfo
from ddopai.agents.obsprocessors import FlattenTimeDimNumpy

import torch 

In [ ]:

class ILQXagent(BaseAgent):

    """
    Base class for greedy bandit agents.
    """

    def __init__(self,
                 environment_info: MDPInfo,
                 obsprocessors: Optional[List[object]] = None,
                 actionprocessors: Optional[List[object]] = None,
                 agent_name: str | None = None,
                 ex_prices: np.ndarray | None = None,
                 alpha: np.ndarray | None = None,
                 beta: np.ndarray | None = None,
                 price_function = None,
                 g = GLMLink,
                 l1_lambda: float = 0.01,
                 lr: float = 0.01,
                 
                 ):
        assert type(alpha) == type(beta), "alpha and beta must be of the same type"
        if type(alpha) == None:
            alpha = np.zeros(environment_info.observation_space.shape[0])
            beta = np.zeros(environment_info.observation_space.shape[0])
        assert ex_prices.shape[0] >= 2
        
        self.ex_prices = ex_prices
        self.alpha = alpha
        self.beta = beta
        self.actionprocessors = actionprocessors
        self.price_function = price_function # Needs to return an np array
        self.lr = lr
        self.g = g
        self.t = 0
        self.l1_lambda = l1_lambda
        super().__init__(environment_info = environment_info, obsprocessors = obsprocessors, agent_name = agent_name)

    def draw_action_(self, observation: np.ndarray):
        if self.t <= self.ex_prices.shape[0]:
            price = self.ex_prices[self.t]
        else:
            price = self.price_function(observation, self.alpha, self.beta)
            
        for processor in self.actionprocessors:
            price = processor(price)
        
        return price
    
    def fit(self, X, Y):
        assert self.mode == "train"
        self.t += 1
        
        self.parameter_update(X, Y)
    
    def parameter_update(self, X, Y):
        
        alpha_tensor = torch.tensor(self.alpha, requires_grad=True)
        beta_tensor = torch.tensor(self.beta, requires_grad=True)
        theta_tensor = torch.cat((alpha_tensor, beta_tensor), requires_grad=True)
        
        optimizer = torch.optim.sgd([alpha_tensor, beta_tensor], lr=self.lr)
        X_tensor = torch.tensor(X)
        Y_tensor = torch.tensor(Y)
        
        optimizer.zero_grad()
        
        loss = self.quasi_loss(theta_tensor, X_tensor, Y_tensor)
        loss.backward()
        optimizer.step()
        
        self.alpha = alpha_tensor.detach().numpy()
        self.beta = beta_tensor.detach().numpy()
        
    def quasi_loss(self, theta, u_k, d_k, num_samples = 100):
        theta_u_k = self.g.g_inv(torch.dot(theta, u_k))
        y_values = torch.linspace(d_k, theta_u_k, num_samples)
        dy = (theta_u_k - d_k) / num_samples
        integrand = (d_k - y_values) / self.g.v(y_values)
        integral_term = torch.sum(integrand) * dy
        l1_penalty_term = self.l1_lambda * torch.sum(torch.abs(theta))
        return -integral_term + l1_penalty_term
        
        